In [3]:
!pip -q install --upgrade openai pandas tqdm

from google.colab import drive
drive.mount("/content/drive")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.6.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass.getpass("OPENAI_API_KEY 입력: ")

os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI

client = OpenAI()

MODEL = "gpt-5.5"

print("OPENAI_API_KEY 설정 완료")
print("client 생성 완료")
print("MODEL:", MODEL)

OPENAI_API_KEY 입력: ··········
OPENAI_API_KEY 설정 완료
client 생성 완료
MODEL: gpt-5.5


In [6]:
# ============================================================
# 자과캠 + 인사캠 category_tag / min_price / max_price / price_range 생성
# GPT-5.5 + web_search 사용
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time
import random
import re
import csv
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/Crawler")
N_DATA_DIR = BASE_DIR / "data" / "n"
H_DATA_DIR = BASE_DIR / "data" / "h"

ENRICH_OUTPUT_COLUMNS = [
    "restaurant_external_id",
    "restaurant_name",
    "campus_slug",
    "category_tag",
    "min_price",
    "max_price",
    "price_range",
]

USE_WEB_SEARCH = True

print("N_DATA_DIR:", N_DATA_DIR)
print("H_DATA_DIR:", H_DATA_DIR)
print("MODEL:", MODEL)

N_DATA_DIR: /content/drive/MyDrive/Crawler/data/n
H_DATA_DIR: /content/drive/MyDrive/Crawler/data/h
MODEL: gpt-5.5


In [7]:
# ============================================================
# 파일 찾기 유틸
# ============================================================

def first_existing(base_dir: Path, candidates: list[str]) -> Path:
    for name in candidates:
        path = base_dir / name
        if path.exists():
            return path
    raise FileNotFoundError(f"{base_dir}에서 다음 후보 파일을 찾지 못함: {candidates}")

def read_csv_keep(path: Path):
    return pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)

def load_campus_inputs(data_dir: Path, campus_slug: str):
    status_path = first_existing(data_dir, [
        "02_review_collection_status.csv",
        "review_collection_status.csv",
        "h_review_collection_status.csv",
        "h_02_review_collection_status.csv",
    ])

    raw_path = first_existing(data_dir, [
        "02_raw_reviews_manual.csv",
        "raw_reviews_manual.csv",
        "h_raw_reviews_manual.csv",
        "h_02_raw_reviews_manual.csv",
    ])

    candidates_path = first_existing(data_dir, [
        "kakao_restaurant_candidates.csv",
        "h_kakao_restaurant_candidates.csv",
    ])

    summary_path = first_existing(data_dir, [
        "restaurant_summaries.csv",
    ])

    status_df = read_csv_keep(status_path)
    raw_df = read_csv_keep(raw_path)
    candidates_df = read_csv_keep(candidates_path)
    summaries_df = read_csv_keep(summary_path)

    print(f"\n[{campus_slug}]")
    print("status:", status_path, status_df.shape)
    print("raw:", raw_path, raw_df.shape)
    print("candidates:", candidates_path, candidates_df.shape)
    print("summaries:", summary_path, summaries_df.shape)

    return {
        "campus_slug": campus_slug,
        "data_dir": data_dir,
        "status_path": status_path,
        "raw_path": raw_path,
        "candidates_path": candidates_path,
        "summary_path": summary_path,
        "status_df": status_df,
        "raw_df": raw_df,
        "candidates_df": candidates_df,
        "summaries_df": summaries_df,
    }

n_inputs = load_campus_inputs(N_DATA_DIR, "natural")
h_inputs = load_campus_inputs(H_DATA_DIR, "humanities")


[natural]
status: /content/drive/MyDrive/Crawler/data/n/review_collection_status.csv (45, 6)
raw: /content/drive/MyDrive/Crawler/data/n/raw_reviews_manual.csv (2250, 5)
candidates: /content/drive/MyDrive/Crawler/data/n/kakao_restaurant_candidates.csv (45, 14)
summaries: /content/drive/MyDrive/Crawler/data/n/restaurant_summaries.csv (45, 8)

[humanities]
status: /content/drive/MyDrive/Crawler/data/h/h_review_collection_status.csv (45, 6)
raw: /content/drive/MyDrive/Crawler/data/h/h_raw_reviews_manual.csv (2250, 5)
candidates: /content/drive/MyDrive/Crawler/data/h/h_kakao_restaurant_candidates.csv (45, 14)
summaries: /content/drive/MyDrive/Crawler/data/h/restaurant_summaries.csv (45, 8)


In [8]:
# ============================================================
# 리뷰/카카오/요약 근거 묶기
# ============================================================

def clean_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\u00a0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def build_reviews_map(raw_df):
    raw_df = raw_df.copy()

    if "review_text" not in raw_df.columns:
        raise ValueError("raw_df에 review_text 컬럼이 없습니다.")

    raw_df["review_text_clean"] = raw_df["review_text"].apply(clean_text)
    raw_nonempty = raw_df[raw_df["review_text_clean"] != ""].copy()

    sort_cols = ["restaurant_external_id"]
    if "review_index" in raw_nonempty.columns:
        sort_cols.append("review_index")

    return (
        raw_nonempty
        .sort_values(sort_cols)
        .groupby("restaurant_external_id")["review_text_clean"]
        .apply(list)
        .to_dict()
    )

def normalize_id(x):
    return str(x).strip()

def attach_candidate_info(status_df, candidates_df):
    status_df = status_df.copy()
    candidates_df = candidates_df.copy()

    status_df["restaurant_external_id"] = status_df["restaurant_external_id"].astype(str)
    status_df["kakao_place_id"] = status_df["kakao_place_id"].astype(str)

    candidates_df["kakao_place_id"] = candidates_df["kakao_place_id"].astype(str)

    candidate_cols = [
        "kakao_place_id",
        "name",
        "category_name",
        "address",
        "road_address",
        "phone",
        "kakao_place_url",
    ]

    candidate_cols = [c for c in candidate_cols if c in candidates_df.columns]

    cand_small = (
        candidates_df[candidate_cols]
        .drop_duplicates(subset=["kakao_place_id"], keep="first")
        .copy()
    )

    merged = status_df.merge(
        cand_small,
        on="kakao_place_id",
        how="left",
        suffixes=("", "_kakao"),
    )

    return merged

def attach_summary_info(base_df, summaries_df):
    base_df = base_df.copy()
    summaries_df = summaries_df.copy()

    base_df["restaurant_external_id"] = base_df["restaurant_external_id"].astype(str)
    summaries_df["restaurant_external_id"] = summaries_df["restaurant_external_id"].astype(str)

    summary_cols = [
        "restaurant_external_id",
        "summary",
        "representative_menu",
        "mood_summary",
        "parking_summary",
        "waiting_summary",
    ]

    summary_cols = [c for c in summary_cols if c in summaries_df.columns]

    return base_df.merge(
        summaries_df[summary_cols],
        on="restaurant_external_id",
        how="left",
    )

def build_enrich_base(inputs):
    status_df = inputs["status_df"]
    candidates_df = inputs["candidates_df"]
    summaries_df = inputs["summaries_df"]
    campus_slug = inputs["campus_slug"]

    base_df = attach_candidate_info(status_df, candidates_df)
    base_df = attach_summary_info(base_df, summaries_df)

    base_df["campus_slug"] = campus_slug

    if "status" in base_df.columns:
        base_df["status_norm"] = base_df["status"].astype(str).str.strip().str.lower()
        base_df = base_df[base_df["status_norm"] == "done"].copy()

    return base_df.reset_index(drop=True)

n_enrich_base = build_enrich_base(n_inputs)
h_enrich_base = build_enrich_base(h_inputs)

n_reviews_map = build_reviews_map(n_inputs["raw_df"])
h_reviews_map = build_reviews_map(h_inputs["raw_df"])

print("자과캠 enrichment 대상:", n_enrich_base.shape)
print("인사캠 enrichment 대상:", h_enrich_base.shape)

display(n_enrich_base.head())
display(h_enrich_base.head())

자과캠 enrichment 대상: (40, 19)
인사캠 enrichment 대상: (37, 19)


,restaurant_external_id,restaurant_name,kakao_place_id,kakao_place_url,collected_review_count,status,name,category_name,address,road_address,phone,kakao_place_url_kakao,summary,representative_menu,mood_summary,parking_summary,waiting_summary,campus_slug,status_norm
0,kakao_824736542,낭만가득한20세기포장마차 수원성대본점,824736542,http://place.map.kakao.com/824736542,11,done,낭만가득한20세기포장마차 수원성대본점,음식점 > 술집 > 실내포장마차,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,,http://place.map.kakao.com/824736542,"안주와 술이 다양하고 맛있다는 평가가 많으며, 신선한 재료와 깔끔한 오픈주방, 넉넉...",꽁치김치찌개 | 생맥주 | 하이볼 | 골뱅이무침 | 명란계란말이 | 김치볶음밥 | ...,"포장마차처럼 가볍지만 운치 있는 분위기이며, 인테리어와 음악 선곡·음량이 좋아 적당...",NAN,NAN,natural,done
1,kakao_306868449,소리산전집,306868449,http://place.map.kakao.com/306868449,19,done,소리산전집,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,,http://place.map.kakao.com/306868449,"전과 식사 메뉴 전반이 맛있고 정갈하다는 평이 많으며, 기본 반찬과 계란후라이가 나...",모듬전 | 육전 | 김치찌개 | 제육덮밥 | 한우스지 | 스지곰탕 | 감자전 | 김치전,"가게는 작지만 아늑하고 깨끗하다는 언급이 있으며, 막걸리와 전을 함께 먹기 좋고 학...",NAN,NAN,natural,done
2,kakao_1279255339,두루정 수원성대점,1279255339,http://place.map.kakao.com/1279255339,50,done,두루정 수원성대점,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,031-293-2007,http://place.map.kakao.com/1279255339,"짜글이와 두루치기를 중심으로 맛있고 양이 푸짐하다는 평가가 많으며, 공기밥 제공·무...",짜글이 | 두루치기 | 제육 파김치 짜글이 | 김치부대찌개 짜글이 | 계란찜 | 쭈...,"가게가 조용하고 아늑하며 분위기가 좋아 데이트나 혼밥하기 좋다는 언급이 있고, 내부...",버스정류소가 가깝고 공영주차장이 도보 2분 거리에 있으며 1시간 무료라는 언급이 있...,NAN,natural,done
3,kakao_762026624,코아김밥,762026624,http://place.map.kakao.com/762026624,20,done,코아김밥,음식점 > 분식,경기 수원시 장안구 율전동 419,경기 수원시 장안구 서부로 2095,031-278-0351,http://place.map.kakao.com/762026624,"김밥 속재료가 푸짐하고 한 줄로도 든든하다는 평이 많으며, 묵은지참치김밥과 치즈김밥...",매운어묵김밥 | 묵은지참치김밥 | 치즈김밥 | 멸추김밥 | 멸치김밥,NAN,NAN,NAN,natural,done
4,kakao_1676208114,최고집 원조 수구레,1676208114,http://place.map.kakao.com/1676208114,50,done,최고집 원조 수구레,음식점 > 한식,경기 수원시 장안구 율전동 433-79,경기 수원시 장안구 서부로 2107,070-8843-4329,http://place.map.kakao.com/1676208114,"수구레 전골, 볶음, 모듬구이 등 수구레 메뉴가 부드럽고 쫄깃하며 냄새 없이 맛있다...",수구레 전골 | 수구레 볶음 | 수구레모듬구이 | 볶음밥 | 곱창구이 | 수육 | ...,"술 한잔이나 반주하기 좋고, 여자친구·가족·지인과 방문했다는 리뷰가 있으며 좌석이 ...",공영주차장이 가까워 편리하다는 리뷰가 있습니다.,"웨이팅이 있지만 회전이 빠른 편이라는 언급이 있고, 초저녁부터 웨이팅이 있으며 늘 ...",natural,done


,restaurant_external_id,restaurant_name,kakao_place_id,kakao_place_url,collected_review_count,status,name,category_name,address,road_address,phone,kakao_place_url_kakao,summary,representative_menu,mood_summary,parking_summary,waiting_summary,campus_slug,status_norm
0,kakao_168487509,인생건어물맥주 대학로점,168487509,http://place.map.kakao.com/168487509,13,done,인생건어물맥주 대학로점,"음식점 > 술집 > 호프,요리주점 > 인생건어물맥주",서울 종로구 명륜2가 122,서울 종로구 성균관로 26,,http://place.map.kakao.com/168487509,"성균관대 바로 앞에 있어 위치가 좋고, 안주와 맥주가 맛있으며 가격도 저렴하다는 평...",오뎅탕 | 마라바지락볶음면 | 감바스 | 먹태 | 살얼음생맥주,"성균관대 앞 풍경을 보며 먹기 좋고, 혼술이나 친구와 대화하며 술 마시기 좋은 분위...",NAN,NAN,humanities,done
1,kakao_11634881,한솥도시락 성균관대앞점,11634881,http://place.map.kakao.com/11634881,50,done,한솥도시락 성균관대앞점,음식점 > 도시락 > 한솥도시락,서울 종로구 명륜2가 122,서울 종로구 성균관로 26,02-744-0762,http://place.map.kakao.com/11634881,"저렴한 가격과 가성비 좋은 요일·행사 메뉴, 맛있는 도시락으로 점심이나 간단한 식사...",불닭치킨마요 | 치즈닭갈비덮밥 | 스팸마요 | 치킨마요 | 순살 후라이드,혼밥하기 좋고 편하게 간단한 식사를 하기 좋은 매장입니다.,NAN,"도시락이나 포장이 빨리 나온다는 리뷰가 있지만, 주문 후 음식이 나오기까지 조금 시...",humanities,done
2,kakao_22016023,맥덕스,22016023,http://place.map.kakao.com/22016023,50,done,맥덕스,음식점 > 양식,서울 종로구 명륜2가 135-3,서울 종로구 성균관로 25-9,070-8623-6397,http://place.map.kakao.com/22016023,"성균관대 앞에서 피자와 맥주 조합을 즐기기 좋은 피맥집으로, 피자 도우와 다양한 수...",페퍼로니 피자 | 비비큐 포크 피자 | 더블 치즈 피자 | 맥앤치즈 | 치즈 오븐 ...,"돌담길 옆 야외 테라스와 야장 감성이 좋고, 아늑하고 조용한 분위기라 데이트나 피맥...",NAN,NAN,humanities,done
3,kakao_338935651,명륜진사갈비 서울성균관대점,338935651,http://place.map.kakao.com/338935651,50,done,명륜진사갈비 서울성균관대점,"음식점 > 한식 > 육류,고기 > 갈비 > 명륜진사갈비",서울 종로구 명륜3가 53,서울 종로구 성균관로 31,02-747-9279,http://place.map.kakao.com/338935651,"고기 질과 양념갈비 맛, 다양한 사이드 메뉴와 가성비에 대한 만족도가 높고 직원·사...",돼지갈비 | 양념갈비 | 돼지껍데기 | 계란찜 | 된장찌개 | 햄버거 | 감자튀김 ...,"매장이 넓고 깔끔하며 단체석이 많아 가족모임, 회식, 친구 모임, 데이트에 이용하기...",주차가 편하고 주차 공간이 넓거나 넉넉하다는 리뷰가 여러 번 언급되었습니다.,"예약 손님이 많아 자리를 못 잡을 뻔했다는 언급과, 주말에는 인원이 있으면 예약이 ...",humanities,done
4,kakao_122821905,물결식당,122821905,http://place.map.kakao.com/122821905,15,done,물결식당,음식점 > 퓨전요리 > 퓨전한식,서울 종로구 명륜2가 135-3,서울 종로구 성균관로 25-9,,http://place.map.kakao.com/122821905,"물결식당은 고등어덮밥, 카레, 닭고기 덮밥 등 덮밥류와 파스타 메뉴가 맛있다는 평가...",고등어덮밥 | 카레 | 쟌슨빌 크림 라이스 | 닭고기 덮밥 | 잔슨빌들깨파스타 | ...,"아담하고 아기자기한 인테리어에 조용하고 따뜻한 분위기이며, 일본 감성이나 연남동 골...",NAN,NAN,humanities,done


In [9]:
# ============================================================
# GPT-5.5 category/price 추출 스키마
# ============================================================

PRICE_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "category_tag": {
            "type": "string",
            "enum": ["한식", "중식", "일식", "양식"],
            "description": "식당의 음식 카테고리. 반드시 4개 중 하나."
        },
        "min_price": {
            "anyOf": [{"type": "integer"}, {"type": "null"}],
            "description": "대표 식사 메뉴 기준 최저 가격. 원 단위 정수. 확인 불가하면 null."
        },
        "max_price": {
            "anyOf": [{"type": "integer"}, {"type": "null"}],
            "description": "대표 식사 메뉴 기준 최고 가격. 원 단위 정수. 확인 불가하면 null."
        },
        "price_range": {
            "type": "string",
            "enum": ["Low", "Mid", "High"],
            "description": "가격대. Low/Mid/High 중 하나."
        },
        "evidence": {
            "type": "string",
            "description": "판단 근거를 짧게 작성. CSV에는 저장하지 않음."
        }
    },
    "required": [
        "category_tag",
        "min_price",
        "max_price",
        "price_range",
        "evidence",
    ],
}

PRICE_SYSTEM_PROMPT = """
너는 성균관대학교 주변 식당 데이터 정제 모듈이다.

목표:
식당명, 카카오 카테고리, 리뷰, 기존 요약, 필요시 웹 검색 결과를 바탕으로
category_tag, min_price, max_price, price_range를 추출한다.

반드시 지킬 규칙:
1. category_tag는 반드시 한식, 중식, 일식, 양식 중 하나만 사용한다.
2. 분식, 국밥, 고기, 찌개, 백반, 치킨, 족발, 보쌈, 술집 안주류는 가장 가까운 경우 한식으로 분류한다.
3. 돈카츠, 라멘, 초밥, 우동, 이자카야는 일식으로 분류한다.
4. 짜장면, 짬뽕, 마라탕, 훠궈, 양꼬치는 중식으로 분류한다.
5. 파스타, 피자, 버거, 샌드위치, 스테이크, 브런치는 양식으로 분류한다.
6. 카페/디저트가 섞여 있더라도 식사 메뉴 중심으로 가장 가까운 4분류를 선택한다.
7. min_price와 max_price는 대표 식사 메뉴 기준 가격이다.
8. 가격은 원 단위 정수로 쓴다. 예: 9000
9. 가격을 확인할 수 없으면 min_price와 max_price는 null로 둔다.
10. 단, price_range는 반드시 Low/Mid/High 중 하나로 판단한다.
11. price_range 기준:
    - Low: 대표 식사 메뉴 최저가가 9000원 이하
    - Mid: 대표 식사 메뉴 최저가가 9001원 이상 15000원 이하
    - High: 대표 식사 메뉴 최저가가 15001원 이상
12. 정확한 가격 확인이 어렵지만 리뷰에서 저렴함/가성비/비쌈이 드러나면 price_range만 판단한다.
13. 근거가 불확실하면 evidence에 불확실하다고 적되, category_tag와 price_range는 반드시 선택한다.
14. 원문에 없는 가격을 확정적으로 지어내지 않는다.
"""

In [10]:
# ============================================================
# GPT 호출 함수
# ============================================================

def safe_value(row, col):
    if col not in row:
        return "NAN"
    x = row.get(col, "NAN")
    if pd.isna(x):
        return "NAN"
    x = str(x).strip()
    if x == "":
        return "NAN"
    return x

def build_price_user_prompt(row, reviews):
    restaurant_name = safe_value(row, "restaurant_name")
    if restaurant_name == "NAN":
        restaurant_name = safe_value(row, "name")

    clipped_reviews = []
    for i, review in enumerate(reviews[:30], start=1):
        review = clean_text(review)
        if len(review) > 300:
            review = review[:300] + "..."
        clipped_reviews.append(f"{i}. {review}")

    reviews_text = "\n".join(clipped_reviews) if clipped_reviews else "NAN"

    return f"""
식당 정보:
restaurant_external_id: {safe_value(row, "restaurant_external_id")}
restaurant_name: {restaurant_name}
campus_slug: {safe_value(row, "campus_slug")}

카카오 정보:
kakao_category_name: {safe_value(row, "category_name")}
address: {safe_value(row, "address")}
road_address: {safe_value(row, "road_address")}
kakao_place_url: {safe_value(row, "kakao_place_url")}

기존 요약:
summary: {safe_value(row, "summary")}
representative_menu: {safe_value(row, "representative_menu")}
mood_summary: {safe_value(row, "mood_summary")}

리뷰 일부:
{reviews_text}

작업:
이 식당의 category_tag, min_price, max_price, price_range를 JSON Schema에 맞게 반환하라.
필요하면 웹 검색을 사용해 메뉴 가격을 확인하라.
"""

def normalize_price_result(data):
    category_tag = str(data.get("category_tag", "")).strip()
    if category_tag not in {"한식", "중식", "일식", "양식"}:
        category_tag = "한식"

    def to_int_or_none(x):
        if x is None:
            return None
        try:
            if pd.isna(x):
                return None
        except Exception:
            pass

        text = str(x)
        text = re.sub(r"[^0-9]", "", text)

        if text == "":
            return None

        return int(text)

    min_price = to_int_or_none(data.get("min_price"))
    max_price = to_int_or_none(data.get("max_price"))

    if min_price is not None and max_price is not None and min_price > max_price:
        min_price, max_price = max_price, min_price

    price_range = str(data.get("price_range", "")).strip()

    if price_range not in {"Low", "Mid", "High"}:
        if min_price is not None:
            if min_price <= 9000:
                price_range = "Low"
            elif min_price <= 15000:
                price_range = "Mid"
            else:
                price_range = "High"
        else:
            price_range = "Mid"

    return {
        "category_tag": category_tag,
        "min_price": min_price,
        "max_price": max_price,
        "price_range": price_range,
    }

def extract_price_profile_with_gpt(row, reviews, max_retries=5):
    user_prompt = build_price_user_prompt(row, reviews)

    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": MODEL,
                "reasoning": {
                    "effort": "low"
                },
                "max_output_tokens": 1200,
                "input": [
                    {
                        "role": "system",
                        "content": PRICE_SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    },
                ],
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "restaurant_price_profile",
                        "strict": True,
                        "schema": PRICE_SCHEMA,
                    }
                },
            }

            if USE_WEB_SEARCH:
                kwargs["tools"] = [
                    {
                        "type": "web_search"
                    }
                ]
                kwargs["tool_choice"] = "auto"

            response = client.responses.create(**kwargs)
            data = json.loads(response.output_text)

            return normalize_price_result(data)

        except Exception as e:
            wait = min(60, (2 ** attempt) + random.random())
            rid = safe_value(row, "restaurant_external_id")
            print(f"[재시도 {attempt + 1}/{max_retries}] {rid}: {e}")
            time.sleep(wait)

    return {
        "category_tag": "한식",
        "min_price": None,
        "max_price": None,
        "price_range": "Mid",
    }

In [11]:
# ============================================================
# 캠퍼스별 enrichment CSV 생성
# ============================================================

def create_enrichment_csv(enrich_base_df, reviews_map, campus_slug, output_path, limit=None):
    target_df = enrich_base_df.copy()

    if limit is not None:
        target_df = target_df.head(limit)

    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=ENRICH_OUTPUT_COLUMNS)
        writer.writeheader()

    print(f"[{campus_slug}] enrichment 생성 시작:", output_path)
    print("대상 식당 수:", len(target_df))

    for _, row in tqdm(target_df.iterrows(), total=len(target_df)):
        restaurant_external_id = str(row["restaurant_external_id"]).strip()

        restaurant_name = safe_value(row, "restaurant_name")
        if restaurant_name == "NAN":
            restaurant_name = safe_value(row, "name")

        reviews = reviews_map.get(restaurant_external_id, [])

        profile = extract_price_profile_with_gpt(row, reviews)

        out_row = {
            "restaurant_external_id": restaurant_external_id,
            "restaurant_name": restaurant_name,
            "campus_slug": campus_slug,
            "category_tag": profile["category_tag"],
            "min_price": profile["min_price"] if profile["min_price"] is not None else "NAN",
            "max_price": profile["max_price"] if profile["max_price"] is not None else "NAN",
            "price_range": profile["price_range"],
        }

        with open(output_path, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=ENRICH_OUTPUT_COLUMNS)
            writer.writerow(out_row)

    print(f"[{campus_slug}] enrichment 생성 완료:", output_path)

    result_df = pd.read_csv(output_path, encoding="utf-8-sig", keep_default_na=False)
    print("shape:", result_df.shape)
    display(result_df.head(20))

    return result_df

In [12]:
# ============================================================
# 전체 실행
# ============================================================

n_enrich_path = N_DATA_DIR / "restaurant_price_category_enrichment.csv"
h_enrich_path = H_DATA_DIR / "restaurant_price_category_enrichment.csv"

n_enrich_df = create_enrichment_csv(
    enrich_base_df=n_enrich_base,
    reviews_map=n_reviews_map,
    campus_slug="natural",
    output_path=n_enrich_path,
    limit=None,
)

h_enrich_df = create_enrichment_csv(
    enrich_base_df=h_enrich_base,
    reviews_map=h_reviews_map,
    campus_slug="humanities",
    output_path=h_enrich_path,
    limit=None,
)

[natural] enrichment 생성 시작: /content/drive/MyDrive/Crawler/data/n/restaurant_price_category_enrichment.csv
대상 식당 수: 40


  0%|          | 0/40 [00:00<?, ?it/s]

[natural] enrichment 생성 완료: /content/drive/MyDrive/Crawler/data/n/restaurant_price_category_enrichment.csv
shape: (40, 7)


,restaurant_external_id,restaurant_name,campus_slug,category_tag,min_price,max_price,price_range
0,kakao_824736542,낭만가득한20세기포장마차 수원성대본점,natural,한식,NAN,NAN,Low
1,kakao_306868449,소리산전집,natural,한식,6000,6000,Low
2,kakao_1279255339,두루정 수원성대점,natural,한식,9500,14500,Mid
3,kakao_762026624,코아김밥,natural,한식,NAN,NAN,Low
4,kakao_1676208114,최고집 원조 수구레,natural,한식,19000,19000,High
5,kakao_1687006483,돼지게티 율천점,natural,양식,15000,20000,Mid
6,kakao_24583309,신전떡볶이 수원성대점,natural,한식,4000,12000,Low
7,kakao_2064565238,중경마라탕 3호점,natural,중식,7000,15000,Low
8,kakao_1665003130,완미족발 수원성대점,natural,한식,37900,45000,High
9,kakao_765114919,텐푸라바사바사,natural,일식,11500,15000,Mid


[humanities] enrichment 생성 시작: /content/drive/MyDrive/Crawler/data/h/restaurant_price_category_enrichment.csv
대상 식당 수: 37


  0%|          | 0/37 [00:00<?, ?it/s]

[humanities] enrichment 생성 완료: /content/drive/MyDrive/Crawler/data/h/restaurant_price_category_enrichment.csv
shape: (37, 7)


,restaurant_external_id,restaurant_name,campus_slug,category_tag,min_price,max_price,price_range
0,kakao_168487509,인생건어물맥주 대학로점,humanities,한식,NAN,NAN,Low
1,kakao_11634881,한솥도시락 성균관대앞점,humanities,한식,3900,7300,Low
2,kakao_22016023,맥덕스,humanities,양식,14000,14000,Mid
3,kakao_338935651,명륜진사갈비 서울성균관대점,humanities,한식,21900,21900,High
4,kakao_122821905,물결식당,humanities,한식,NAN,NAN,Low
5,kakao_25770908,피자헤븐 종로성대점,humanities,양식,22900,22900,High
6,kakao_1869234515,맘스터치 피자앤치킨 종로성대점,humanities,양식,10900,24900,Mid
7,kakao_483030266,명분,humanities,양식,NAN,NAN,Low
8,kakao_633176459,집시포차 혜화점,humanities,한식,NAN,NAN,Mid
9,kakao_11520636,페르시안궁전,humanities,양식,12000,43000,Mid


In [13]:
# ============================================================
# 자과캠 + 인사캠 가격/카테고리 보강 합본 생성
# ============================================================

all_enrich_path = BASE_DIR / "data" / "all_restaurant_price_category_enrichment.csv"

n_enrich_df = pd.read_csv(
    N_DATA_DIR / "restaurant_price_category_enrichment.csv",
    encoding="utf-8-sig",
    keep_default_na=False,
)

h_enrich_df = pd.read_csv(
    H_DATA_DIR / "restaurant_price_category_enrichment.csv",
    encoding="utf-8-sig",
    keep_default_na=False,
)

all_enrich_df = pd.concat(
    [n_enrich_df, h_enrich_df],
    ignore_index=True,
)

all_enrich_df = all_enrich_df[ENRICH_OUTPUT_COLUMNS].copy()

all_enrich_df = all_enrich_df.drop_duplicates(
    subset=["restaurant_external_id"],
    keep="first",
).reset_index(drop=True)

all_enrich_df.to_csv(
    all_enrich_path,
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료:", all_enrich_path)
print("shape:", all_enrich_df.shape)

display(all_enrich_df.head(50))

저장 완료: /content/drive/MyDrive/Crawler/data/all_restaurant_price_category_enrichment.csv
shape: (77, 7)


,restaurant_external_id,restaurant_name,campus_slug,category_tag,min_price,max_price,price_range
0,kakao_824736542,낭만가득한20세기포장마차 수원성대본점,natural,한식,NAN,NAN,Low
1,kakao_306868449,소리산전집,natural,한식,6000,6000,Low
2,kakao_1279255339,두루정 수원성대점,natural,한식,9500,14500,Mid
3,kakao_762026624,코아김밥,natural,한식,NAN,NAN,Low
4,kakao_1676208114,최고집 원조 수구레,natural,한식,19000,19000,High
5,kakao_1687006483,돼지게티 율천점,natural,양식,15000,20000,Mid
6,kakao_24583309,신전떡볶이 수원성대점,natural,한식,4000,12000,Low
7,kakao_2064565238,중경마라탕 3호점,natural,중식,7000,15000,Low
8,kakao_1665003130,완미족발 수원성대점,natural,한식,37900,45000,High
9,kakao_765114919,텐푸라바사바사,natural,일식,11500,15000,Mid


In [14]:
# ============================================================
# enrichment 검증
# ============================================================

check_df = pd.read_csv(
    BASE_DIR / "data" / "all_restaurant_price_category_enrichment.csv",
    encoding="utf-8-sig",
    keep_default_na=False,
)

print("shape:", check_df.shape)
print("columns:", check_df.columns.tolist())

expected_cols = ENRICH_OUTPUT_COLUMNS

if check_df.columns.tolist() != expected_cols:
    raise RuntimeError(f"컬럼 불일치: {check_df.columns.tolist()}")

dup_count = check_df["restaurant_external_id"].duplicated().sum()
print("restaurant_external_id 중복 수:", dup_count)

if dup_count != 0:
    raise RuntimeError("restaurant_external_id 중복 있음")

valid_categories = {"한식", "중식", "일식", "양식"}
bad_category_df = check_df[
    ~check_df["category_tag"].isin(valid_categories)
]

print("카테고리 분포:")
print(check_df["category_tag"].value_counts(dropna=False))

if not bad_category_df.empty:
    print("잘못된 category_tag:")
    display(bad_category_df)
    raise RuntimeError("category_tag 이상값 있음")

valid_price_ranges = {"Low", "Mid", "High"}
bad_price_df = check_df[
    ~check_df["price_range"].isin(valid_price_ranges)
]

print("\n가격대 분포:")
print(check_df["price_range"].value_counts(dropna=False))

if not bad_price_df.empty:
    print("잘못된 price_range:")
    display(bad_price_df)
    raise RuntimeError("price_range 이상값 있음")

for col in ["min_price", "max_price"]:
    non_nan = check_df[
        ~check_df[col].astype(str).str.strip().isin(["", "NAN", "nan", "NaN", "None", "null"])
    ].copy()

    non_nan[col] = pd.to_numeric(non_nan[col], errors="coerce")

    bad_numeric = non_nan[
        non_nan[col].isna() | (non_nan[col] < 0) | (non_nan[col] > 200000)
    ]

    print(f"\n{col} 입력 수:", len(non_nan))
    print(f"{col} 이상값 수:", len(bad_numeric))

    if not bad_numeric.empty:
        display(bad_numeric)
        raise RuntimeError(f"{col} 이상값 있음")

print("\n검증 통과")
display(check_df.head(20))

shape: (77, 7)
columns: ['restaurant_external_id', 'restaurant_name', 'campus_slug', 'category_tag', 'min_price', 'max_price', 'price_range']
restaurant_external_id 중복 수: 0
카테고리 분포:
category_tag
한식    52
양식    10
중식     9
일식     6
Name: count, dtype: int64

가격대 분포:
price_range
Low     39
Mid     26
High    12
Name: count, dtype: int64

min_price 입력 수: 49
min_price 이상값 수: 0

max_price 입력 수: 45
max_price 이상값 수: 0

검증 통과


,restaurant_external_id,restaurant_name,campus_slug,category_tag,min_price,max_price,price_range
0,kakao_824736542,낭만가득한20세기포장마차 수원성대본점,natural,한식,NAN,NAN,Low
1,kakao_306868449,소리산전집,natural,한식,6000,6000,Low
2,kakao_1279255339,두루정 수원성대점,natural,한식,9500,14500,Mid
3,kakao_762026624,코아김밥,natural,한식,NAN,NAN,Low
4,kakao_1676208114,최고집 원조 수구레,natural,한식,19000,19000,High
5,kakao_1687006483,돼지게티 율천점,natural,양식,15000,20000,Mid
6,kakao_24583309,신전떡볶이 수원성대점,natural,한식,4000,12000,Low
7,kakao_2064565238,중경마라탕 3호점,natural,중식,7000,15000,Low
8,kakao_1665003130,완미족발 수원성대점,natural,한식,37900,45000,High
9,kakao_765114919,텐푸라바사바사,natural,일식,11500,15000,Mid


In [15]:
# ============================================================
# enrichment 결과를 restaurant_tags.csv에 반영
# - category_tag -> tag_type=food
# - price_range -> tag_type=price
# ============================================================

def append_enrichment_tags(data_dir: Path):
    tags_path = data_dir / "restaurant_tags.csv"
    enrich_path = data_dir / "restaurant_price_category_enrichment.csv"

    assert tags_path.exists(), f"restaurant_tags.csv 없음: {tags_path}"
    assert enrich_path.exists(), f"enrichment csv 없음: {enrich_path}"

    tags_df = pd.read_csv(tags_path, encoding="utf-8-sig", keep_default_na=False)
    enrich_df = pd.read_csv(enrich_path, encoding="utf-8-sig", keep_default_na=False)

    required_tag_cols = [
        "restaurant_external_id",
        "tag_type",
        "tag_value",
        "confidence_score",
    ]

    for col in required_tag_cols:
        assert col in tags_df.columns, f"{tags_path}에 {col} 없음"

    extra_rows = []

    price_label_map = {
        "Low": "저렴함",
        "Mid": "보통",
        "High": "비쌈",
    }

    for _, row in enrich_df.iterrows():
        rid = str(row["restaurant_external_id"]).strip()

        category_tag = str(row["category_tag"]).strip()
        price_range = str(row["price_range"]).strip()

        if category_tag in {"한식", "중식", "일식", "양식"}:
            extra_rows.append({
                "restaurant_external_id": rid,
                "tag_type": "food",
                "tag_value": category_tag,
                "confidence_score": 0.95,
            })

        if price_range in price_label_map:
            extra_rows.append({
                "restaurant_external_id": rid,
                "tag_type": "price",
                "tag_value": price_label_map[price_range],
                "confidence_score": 0.90,
            })

    extra_df = pd.DataFrame(extra_rows)

    merged_tags_df = pd.concat(
        [tags_df, extra_df],
        ignore_index=True,
    )

    merged_tags_df["restaurant_external_id"] = merged_tags_df["restaurant_external_id"].astype(str)
    merged_tags_df["tag_type"] = merged_tags_df["tag_type"].astype(str)
    merged_tags_df["tag_value"] = merged_tags_df["tag_value"].astype(str)

    merged_tags_df = merged_tags_df.drop_duplicates(
        subset=["restaurant_external_id", "tag_type", "tag_value"],
        keep="first",
    ).reset_index(drop=True)

    merged_tags_df = merged_tags_df[required_tag_cols]

    backup_path = data_dir / "restaurant_tags_before_enrichment_backup.csv"
    tags_df.to_csv(backup_path, index=False, encoding="utf-8-sig")

    merged_tags_df.to_csv(tags_path, index=False, encoding="utf-8-sig")

    print("백업 저장:", backup_path)
    print("태그 업데이트 완료:", tags_path)
    print("기존 태그 수:", len(tags_df))
    print("추가 후 태그 수:", len(merged_tags_df))

    display(merged_tags_df.tail(30))

    return merged_tags_df

n_tags_updated = append_enrichment_tags(N_DATA_DIR)
h_tags_updated = append_enrichment_tags(H_DATA_DIR)

백업 저장: /content/drive/MyDrive/Crawler/data/n/restaurant_tags_before_enrichment_backup.csv
태그 업데이트 완료: /content/drive/MyDrive/Crawler/data/n/restaurant_tags.csv
기존 태그 수: 384
추가 후 태그 수: 464


,restaurant_external_id,tag_type,tag_value,confidence_score
434,kakao_1578043317,food,한식,0.95
435,kakao_1578043317,price,보통,0.90
436,kakao_18829773,food,한식,0.95
437,kakao_18829773,price,비쌈,0.90
438,kakao_148202780,food,일식,0.95
439,kakao_148202780,price,보통,0.90
440,kakao_1673819877,food,한식,0.95
441,kakao_1673819877,price,보통,0.90
442,kakao_1328628226,food,한식,0.95
443,kakao_1328628226,price,저렴함,0.90


AssertionError: restaurant_tags.csv 없음: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv

In [16]:
# 인사캠 restaurant_tags.csv 생성 여부 확인
print("H tags exists:", (H_DATA_DIR / "restaurant_tags.csv").exists())
print("H enrich exists:", (H_DATA_DIR / "restaurant_price_category_enrichment.csv").exists())

print("H tags path:", H_DATA_DIR / "restaurant_tags.csv")
print("H enrich path:", H_DATA_DIR / "restaurant_price_category_enrichment.csv")

H tags exists: True
H enrich exists: True
H tags path: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv
H enrich path: /content/drive/MyDrive/Crawler/data/h/restaurant_price_category_enrichment.csv


In [17]:
h_tags_updated = append_enrichment_tags(H_DATA_DIR)

백업 저장: /content/drive/MyDrive/Crawler/data/h/restaurant_tags_before_enrichment_backup.csv
태그 업데이트 완료: /content/drive/MyDrive/Crawler/data/h/restaurant_tags.csv
기존 태그 수: 357
추가 후 태그 수: 431


,restaurant_external_id,tag_type,tag_value,confidence_score
401,kakao_16036704,food,한식,0.95
402,kakao_16036704,price,비쌈,0.90
403,kakao_9968170,food,중식,0.95
404,kakao_9968170,price,저렴함,0.90
405,kakao_2142464681,food,중식,0.95
406,kakao_2142464681,price,보통,0.90
407,kakao_18889918,food,한식,0.95
408,kakao_18889918,price,저렴함,0.90
409,kakao_1700062133,food,한식,0.95
410,kakao_1700062133,price,저렴함,0.90
